# Domain E: Functional Connectivity and Circuit Interactions

This notebook addresses five questions:
- **E1:** Do noise correlations reveal cell-type-specific connectivity motifs?
- **E2:** Can transfer entropy / Granger causality reveal directed functional interactions?
- **E3:** How does population coupling relate to cell type and spatial position?
- **E4:** Does spontaneous activity structure during grey-screen epochs reveal cell-type-specific network dynamics? *(Zarr data)*
- **E5:** Do catch-trial responses reveal cell-type-specific expectation signals? *(Zarr data)*

**Data:** Zarr multimodal stores with ΔF/F traces (cells × trials), 3D coordinates, cell-type labels, gene expression, spontaneous activity, catch trials, and GLM data.

**Note:** E2 (Granger causality) requires continuous time-series data. The current dataset stores trial-level responses. Sections requiring continuous ΔF/F traces are marked `# ⚠️ REQUIRES: continuous time-series data`.

In [ ]:
import sys
sys.path.insert(0, '.')

import numpy as np
import pandas as pd
import zarr
import matplotlib.pyplot as plt
import seaborn as sns
from types import SimpleNamespace
from scipy.stats import kruskal, spearmanr, pearsonr, mannwhitneyu
from scipy.spatial.distance import cdist, pdist, squareform
import warnings
warnings.filterwarnings('ignore')

from functions.data_loading import load_mouse_zarr, load_zarr_10hz
from functions.analysis import xcorr_pair, pairwise_granger

sns.set_context('talk')
sns.set_style('whitegrid')

# ══════════════════════════════════════════════════════════════════════
# Load data from zarr multimodal stores
# ══════════════════════════════════════════════════════════════════════
ZARR_DIR = 'multimodal_data'
MOUSE_IDS = ['778174', '786297', '797371']
SESSIONS = ['session_1', 'session_2', 'session_3']

adata_list = [load_mouse_zarr(mid, zarr_dir=ZARR_DIR, include_genes=True) for mid in MOUSE_IDS]

obs = pd.concat([a.obs for a in adata_list], ignore_index=True)
X = np.vstack([a.X for a in adata_list])
var = adata_list[0].var.copy()

SUBCLASS_ORDER = ['007 L2/3 IT CTX Glut', '006 L4/5 IT CTX Glut', '022 L5 ET CTX Glut',
                  '052 Pvalb Gaba', '053 Sst Gaba', '046 Vip Gaba', '049 Lamp5 Gaba']
SUBCLASS_COLORS = {'007 L2/3 IT CTX Glut': '#1f77b4', '006 L4/5 IT CTX Glut': '#2ca02c',
                   '022 L5 ET CTX Glut': '#9467bd', '052 Pvalb Gaba': '#d62728',
                   '053 Sst Gaba': '#ff7f0e', '046 Vip Gaba': '#e377c2', '049 Lamp5 Gaba': '#8c564b'}
SUBCLASS_SHORT = {'007 L2/3 IT CTX Glut': 'L2/3 IT', '006 L4/5 IT CTX Glut': 'L4/5 IT',
                  '022 L5 ET CTX Glut': 'L5 ET', '052 Pvalb Gaba': 'Pvalb',
                  '053 Sst Gaba': 'Sst', '046 Vip Gaba': 'Vip', '049 Lamp5 Gaba': 'Lamp5'}
present_subclasses = [s for s in SUBCLASS_ORDER if s in obs['subclass_name'].unique()]

# Backward-compatible adata object for downstream cells
adata = SimpleNamespace(X=X, obs=obs, var=var, n_obs=len(obs), n_vars=X.shape[1])

coords = obs[['x_loc', 'y_loc', 'z_loc']].values.astype(float)
orientations = np.array([0, 45, 90, 135, 180, 225, 270, 315])

# Gene columns
META_COLS = {'unique_id', 'mouse_id', 'class_name', 'subclass_name', 'subclass_label',
             'supertype_name', 'cluster_name', 'cluster_alias', 'x_loc', 'y_loc', 'z_loc'}
GENE_COLS = [c for c in obs.columns if c not in META_COLS]

print(f"Total cells: {len(obs)}, Trials: {X.shape[1]}, Genes: {len(GENE_COLS)}")

## E3: Population Coupling

Measure how strongly each cell's activity correlates with the population mean, split by running state and subclass. Relate coupling to spatial position and gene expression.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# E3.1  Population coupling index per cell
# ══════════════════════════════════════════════════════════════════════

# For each mouse, compute coupling as correlation between each cell's
# trial responses and the population mean (excluding that cell)

run_mask_t = var['is_running'].values.astype(bool)
coupling_all = np.full(adata.n_obs, np.nan)
coupling_run = np.full(adata.n_obs, np.nan)
coupling_stat = np.full(adata.n_obs, np.nan)

for mouse in mouse_ids:
    m_mask = obs['mouse_id'].values == mouse
    m_X = X[m_mask]
    n_m = m_mask.sum()
    m_indices = np.where(m_mask)[0]
    
    # Population mean (excluding self) — efficiently computed
    pop_sum = np.nansum(m_X, axis=0)
    
    for i in range(n_m):
        # Population mean excluding this cell
        pop_exc = (pop_sum - m_X[i]) / (n_m - 1)
        
        # Overall coupling
        valid = ~np.isnan(m_X[i]) & ~np.isnan(pop_exc)
        if valid.sum() > 20:
            coupling_all[m_indices[i]], _ = pearsonr(m_X[i, valid], pop_exc[valid])
        
        # Running trials only
        run_valid = valid & run_mask_t
        if run_valid.sum() > 10:
            coupling_run[m_indices[i]], _ = pearsonr(m_X[i, run_valid], pop_exc[run_valid])
        
        # Stationary trials only
        stat_valid = valid & ~run_mask_t
        if stat_valid.sum() > 10:
            coupling_stat[m_indices[i]], _ = pearsonr(m_X[i, stat_valid], pop_exc[stat_valid])

coupling_df = pd.DataFrame({
    'coupling': coupling_all,
    'coupling_run': coupling_run,
    'coupling_stat': coupling_stat,
    'subclass': obs['subclass_name'].values,
    'subclass_short': obs['subclass_name'].map(SUBCLASS_SHORT).values,
    'x': coords[:, 0],
    'y': coords[:, 1],
    'z': coords[:, 2],
    'mouse_id': obs['mouse_id'].values,
})

print("Population coupling by subclass:")
print(coupling_df.groupby('subclass_short')['coupling'].describe().round(3))

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# E3.2  Visualization: Coupling by subclass, running state, and space
# ══════════════════════════════════════════════════════════════════════

short_order = [SUBCLASS_SHORT[s] for s in present_subclasses]
short_pal = {SUBCLASS_SHORT[k]: v for k, v in SUBCLASS_COLORS.items()}

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Violin: coupling by subclass
ax = axes[0, 0]
sns.violinplot(data=coupling_df[coupling_df['subclass'].isin(present_subclasses)],
               x='subclass_short', y='coupling', order=short_order,
               palette=short_pal, cut=0, inner='box', ax=ax)
ax.axhline(0, color='k', ls='--', alpha=0.4)
ax.set_title('E3: Population Coupling by Subclass', fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Coupling (Pearson r)')
ax.tick_params(axis='x', rotation=45)

# Running vs Stationary coupling
ax = axes[0, 1]
melt = coupling_df[coupling_df['subclass'].isin(present_subclasses)].melt(
    id_vars=['subclass_short'], value_vars=['coupling_run', 'coupling_stat'],
    var_name='state', value_name='coupling_val')
melt['state'] = melt['state'].map({'coupling_run': 'Running', 'coupling_stat': 'Stationary'})
sns.boxplot(data=melt, x='subclass_short', y='coupling_val', hue='state',
            order=short_order, palette={'Running': 'red', 'Stationary': 'blue'}, ax=ax)
ax.set_title('E3: Coupling by Running State', fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Coupling')
ax.legend(fontsize=8)
ax.tick_params(axis='x', rotation=45)
ax.axhline(0, color='k', ls='--', alpha=0.4)

# Coupling vs depth
ax = axes[1, 0]
for sc in present_subclasses:
    mask = coupling_df['subclass'] == sc
    valid = mask & coupling_df['coupling'].notna()
    ax.scatter(coupling_df.loc[valid, 'z'], coupling_df.loc[valid, 'coupling'],
               alpha=0.3, s=10, c=SUBCLASS_COLORS[sc], label=SUBCLASS_SHORT[sc])
ax.set_xlabel('Depth (µm)')
ax.set_ylabel('Population Coupling')
ax.set_title('E3: Coupling vs Cortical Depth', fontweight='bold')
ax.legend(fontsize=7, loc='upper right')
ax.axhline(0, color='k', ls='--', alpha=0.4)

# Spatial map of coupling
ax = axes[1, 1]
valid_c = coupling_df['coupling'].notna()
sc = ax.scatter(coupling_df.loc[valid_c, 'x'], coupling_df.loc[valid_c, 'y'],
                c=coupling_df.loc[valid_c, 'coupling'], cmap='coolwarm', s=10, alpha=0.6,
                vmin=-0.3, vmax=0.3)
plt.colorbar(sc, ax=ax, label='Coupling')
ax.set_xlabel('X (µm)')
ax.set_ylabel('Y (µm)')
ax.set_title('E3: Spatial Map of Population Coupling', fontweight='bold')
ax.set_aspect('equal')

plt.tight_layout()
plt.show()

# ── Kruskal-Wallis ──
groups = [coupling_df.loc[coupling_df['subclass'] == s, 'coupling'].dropna().values
          for s in present_subclasses]
groups = [g for g in groups if len(g) >= 3]
stat, p = kruskal(*groups)
print(f"Coupling Kruskal-Wallis: H={stat:.2f}, p={p:.2e}")

# ── Coupling vs depth (Spearman) per subclass ──
print("\nCoupling vs depth:")
for sc in present_subclasses:
    mask = (coupling_df['subclass'] == sc) & coupling_df['coupling'].notna()
    if mask.sum() >= 10:
        r, p = spearmanr(coupling_df.loc[mask, 'z'], coupling_df.loc[mask, 'coupling'])
        print(f"  {SUBCLASS_SHORT[sc]:10s}: ρ={r:.3f}, p={p:.2e}")